In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# from langchain.retrievers.multi_query import MultiQueryRetriever

from langchain_classic.retrievers import MultiQueryRetriever

from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
    LLMChainExtractor,
)

from langchain_classic.retrievers import ContextualCompressionRetriever

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

from langchain_core.prompts import ChatPromptTemplate

In [3]:
loader = TextLoader("enterprise_data.txt")

docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = splitter.split_documents(docs)

In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [7]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="enterprise_chroma"
)

In [8]:
vector_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":20
    }
)

In [9]:
bm25 = BM25Retriever.from_documents(
    splits
)

bm25.k = 20

In [10]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25,
        vector_retriever
    ],
    weights=[
        0.4,
        0.6
    ]
)

In [11]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5738.27it/s]


In [12]:
rerank_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_retriever,
    base_compressor=reranker
)

In [13]:
compression_llm = ChatOllama(
    model="mistral",
    temperature=0
)

extractor = LLMChainExtractor.from_llm(
    compression_llm
)

In [14]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=rerank_retriever,
    base_compressor=extractor
)

In [15]:
query_llm = ChatOllama(
    model="mistral",
    temperature=0
)

multi_query = MultiQueryRetriever.from_llm(
    retriever=compression_retriever,
    llm=query_llm
)

In [16]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [17]:
prompt = ChatPromptTemplate.from_template("""
You are an Enterprise AI Assistant.

Answer ONLY from the supplied context.

If the answer is unavailable, say:

"I don't know based on the provided documents."

Context:
{context}

Question:
{input}
""")

In [18]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    multi_query,
    document_chain
)

In [19]:
query = "What are the specific tax exemption clauses for 2024?"

response = rag_chain.invoke(
    {
        "input":query
    }
)

print(response["answer"])

 Based on the provided documents, I don't have specific details about the tax exemption clauses effective for the fiscal year 2024. The context only mentions two tax exemptions applicable in 2024: Section 80JJAA (employment generation) with a limit of ₹12 crores and R&D tax benefits under Section 35(2AB) with a deduction of ₹24 crores. For more detailed information, I would recommend consulting a tax professional or the official Indian government taxation website.


In [20]:
query = "What is the maternity leave policy and how does it compare to legal requirements?"

response = rag_chain.invoke(
    {
        "input":query
    }
)

print(response["answer"])

 The maternity leave policy for our company is 26 weeks, which aligns with the legal requirements as per the Maternity Benefit Act 2017.
